# Modelado — Regresión lineal y Random Forest

**Curso:** Modelos y Simulación de Sistemas I — Proyecto integrador, Fase 1
**Continuación de:** `01-exploracion-datos.ipynb` y `02-preparacion-datos.ipynb`

**Problema:** predecir la duración de un viaje (`duracion_viaje_horas`, en horas) hasta la Terminal de Transporte de Medellín, usando información conocida al momento de la salida. Es un problema de **regresión**.

**Qué hace este notebook:** entrena dos modelos con los datos de entrenamiento, una **regresión lineal** y un **Random Forest**, y guarda cada uno en `models/`. La evaluación (validación, métricas en el conjunto de prueba y comparación) se hace en un notebook aparte, para mantener separado el *cómo se construyen* los modelos del *qué tan bien funcionan*.

## Configuración e imports

In [1]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

SEED = 42

TRAIN_PATH = "../data/processed/train.csv.gz"
TEST_PATH = "../data/processed/test.csv.gz"
MODELO_LINEAL_PATH = "../models/regresion_lineal.joblib"
MODELO_RF_PATH = "../models/random_forest.joblib"

OBJETIVO = "duracion_viaje_horas"

## 1. Cargar los datos

Se leen los conjuntos generados en `02-preparacion-datos.ipynb`. La separación ya está hecha: **80 % entrenamiento y 20 % prueba**, aleatoria, con semilla 42. En este notebook el conjunto de prueba **no interviene en el entrenamiento**: solo se usa al final para comprobar que los modelos guardados pueden generar predicciones.

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"train: {train.shape}  |  test: {test.shape}")
print(f"Valores faltantes -> train: {int(train.isna().sum().sum())} | test: {int(test.isna().sum().sum())}")
train.head()

train: (2093700, 11)  |  test: (523425, 11)
Valores faltantes -> train: 0 | test: 0


,nombre_sucursal,empresa,ruta_origen,subregi_n,clase_veh_culo,pasajeros,hora_salida_decimal,dia_semana,mes,duracion_viaje_horas,duracion_corregida
0,TERMINAL DEL NORTE,EXPRE - BELMIRA S.A.,SAN PEDRO DE LOS MILAGROS,NORTE,CAMIONETA,8,10.000000,4,2,1.150000,False
1,TERMINAL DEL NORTE,COONORTE,PUERTO BERRIO,NORTE,MICROBUS,5,4.750000,2,10,3.600000,False
2,TERMINAL DEL NORTE,TRANS CISNEROS-ENTRERRIOS,MACEO,NORTE,BUS,12,8.000000,4,2,2.600000,False
3,TERMINAL DEL NORTE,TRANS CISNEROS-ENTRERRIOS,ENTRERRIOS,NORTE,CAMIONETA,6,5.500000,0,1,2.083333,False
4,TERMINAL DEL NORTE,FLOTA EL CARMEN,EL CARMEN DE VIBORAL,ORIENTE,MICROBUS,19,7.416667,1,9,1.600000,False


## 2. Definir las variables

**Hora de salida.** La exploración mostró que la hora afecta la duración de forma *no lineal*: los viajes que salen de noche duran mucho más que los del día. Una regresión lineal que reciba la hora como un número solo puede aprender "a más hora, más (o menos) duración", lo que no describe ese patrón. Por eso se crea también `hora_entera` (0 a 23), que la regresión lineal tratará como una categoría. Es un cálculo directo sobre cada fila, sin aprender nada de los datos, así que no genera fuga de información. El Random Forest usará la hora original.

**Qué queda fuera.** `duracion_corregida` indica si el objetivo fue reemplazado por la mediana de su ruta; se deriva del objetivo, así que **no puede ser predictora**.

Ambos modelos reciben las **mismas columnas de entrada**, y cada uno selecciona y transforma las que necesita.

In [3]:
for df in (train, test):
    df["hora_entera"] = df["hora_salida_decimal"].astype(int)

CATEGORICAS = ["nombre_sucursal", "empresa", "ruta_origen", "subregi_n", "clase_veh_culo"]
NUMERICAS = ["pasajeros", "hora_salida_decimal", "dia_semana", "mes"]
PREDICTORAS = CATEGORICAS + NUMERICAS + ["hora_entera"]

X_train, y_train = train[PREDICTORAS], train[OBJETIVO]
X_test = test[PREDICTORAS]

# Comprobaciones de seguridad: si alguien las rompe, el notebook falla
assert OBJETIVO not in X_train.columns
assert "duracion_corregida" not in X_train.columns and "duracion_original" not in X_train.columns
assert list(X_train.columns) == list(X_test.columns)

print("Predictoras:", PREDICTORAS)

Predictoras: ['nombre_sucursal', 'empresa', 'ruta_origen', 'subregi_n', 'clase_veh_culo', 'pasajeros', 'hora_salida_decimal', 'dia_semana', 'mes', 'hora_entera']


## 3. Modelo 1 — Regresión lineal

Es un modelo simple, rápido y fácil de interpretar: cada coeficiente indica cuántas horas suma o resta una categoría. Su limitación es que solo aprende relaciones lineales con lo que se le entrega.

**Codificación de las categóricas (one-hot).** La regresión lineal solo trabaja con números. Numerar las categorías (ruta 1, ruta 2, ruta 3...) inventaría un orden que no existe. Con `OneHotEncoder` cada categoría se convierte en una columna de 0 y 1 ("¿el viaje es de esta ruta?"), y el modelo aprende un efecto por categoría. Con `handle_unknown="ignore"`, una categoría que no apareció en entrenamiento no rompe la predicción. Se codifican las 5 categóricas, `hora_entera`, `dia_semana` y `mes` (estos dos últimos son códigos, no cantidades). `pasajeros` ya es numérica y pasa sin cambios.

**Por qué un `Pipeline`.** Une la codificación y el modelo en un solo objeto. Al llamar a `fit`, el codificador aprende las categorías **solo de los datos de entrenamiento**; con datos nuevos únicamente se aplica la transformación ya aprendida. Además, el modelo guardado recibe las columnas originales y devuelve la predicción, sin repetir pasos de preparación.

In [4]:
def crear_regresion_lineal():
    codificacion = ColumnTransformer(
        [
            ("categoricas", OneHotEncoder(handle_unknown="ignore"),
             CATEGORICAS + ["hora_entera", "dia_semana", "mes"]),
            ("pasajeros", "passthrough", ["pasajeros"]),
        ],
        remainder="drop",     # descarta hora_salida_decimal, que este modelo no usa
    )
    return Pipeline([
        ("codificacion", codificacion),
        ("regresion", LinearRegression()),
    ])

modelo_lineal = crear_regresion_lineal()
modelo_lineal

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('codificacion', ...), ('regresion', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categoricas', ...), ('pasajeros', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float,

In [5]:
# Entrenamiento con TODO el conjunto de entrenamiento 
modelo_lineal.fit(X_train, y_train)

n_coef = len(modelo_lineal.named_steps["regresion"].coef_)
print(f"Regresión lineal entrenada con {len(X_train):,} filas y {n_coef} coeficientes.")

Regresión lineal entrenada con 2,093,700 filas y 429 coeficientes.


## 4. Modelo 2 — Random Forest

Un Random Forest es un conjunto de muchos **árboles de decisión**. Cada árbol aprende reglas del tipo "si la ruta es X y la salida es después de las 20:00, entonces la duración es Y", y la predicción final es el **promedio** de todos los árboles. A diferencia de la regresión lineal, puede capturar por sí solo efectos no lineales e interacciones (por ejemplo, que la hora pese distinto según la ruta), sin que se lo preparemos a mano.

**Preprocesamiento.** Los árboles dividen los datos con umbrales, así que no necesitan one-hot ni `hora_entera`:
- Las 5 categóricas se convierten en números con `OrdinalEncoder` (a cada categoría se le asigna un entero). Aunque ese orden es arbitrario, un árbol puede hacer varias divisiones seguidas y aislar cada categoría. Las categorías nuevas reciben el valor -1.
- Las variables numéricas (`pasajeros`, `hora_salida_decimal`, `dia_semana`, `mes`) pasan sin cambios: el árbol encuentra solo los rangos de hora que importan.

**Parámetros elegidos:**

| Parámetro | Valor | Por qué |
|---|---|---|
| `n_estimators` | 80 | Número de árboles. Más árboles dan predicciones más estables, pero el archivo crece con cada uno |
| `min_samples_leaf` | 50 | Cada hoja debe representar al menos 50 viajes. Limita el tamaño de cada árbol (el archivo guardado debe caber en GitHub, cuyo límite es 100 MB) y evita que memorice viajes individuales |
| `max_samples` | 0.5 | Cada árbol se entrena con la mitad de las filas, elegidas al azar. Acelera el entrenamiento y hace que los árboles sean más distintos entre sí |
| `n_jobs` | -1 | Usa todos los núcleos del computador |
| `random_state` | 42 | Semilla, para poder repetir el entrenamiento |

In [6]:
def crear_random_forest():
    codificacion = ColumnTransformer(
        [
            ("categoricas", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAS),
            ("numericas", "passthrough", NUMERICAS),
        ],
        remainder="drop",     # descarta hora_entera, que este modelo no usa
    )
    return Pipeline([
        ("codificacion", codificacion),
        ("bosque", RandomForestRegressor(
            n_estimators=80,
            min_samples_leaf=50,
            max_samples=0.5,
            n_jobs=-1,
            random_state=SEED,
        )),
    ])

modelo_rf = crear_random_forest()
modelo_rf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('codificacion', ...), ('bosque', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categoricas', ...), ('numericas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, de

In [7]:
# Entrenamiento con TODO el conjunto de entrenamiento
modelo_rf.fit(X_train, y_train)

arboles = modelo_rf.named_steps["bosque"].estimators_
nodos = [a.tree_.node_count for a in arboles]
print(f"Random Forest entrenado: {len(arboles)} árboles, {int(np.mean(nodos)):,} nodos por árbol en promedio.")

Random Forest entrenado: 80 árboles, 24,288 nodos por árbol en promedio.


## 5. Guardar los modelos y verificarlos

Cada `Pipeline` completo (preprocesamiento + modelo) se guarda con `joblib`. Se comprueba que:
1. El archivo del Random Forest **cabe en GitHub** (límite de 100 MB por archivo; con más de 50 MB ya aparece una advertencia).
2. Al volver a cargar cada modelo, **genera exactamente las mismas predicciones** que el modelo en memoria.

Para esta comprobación se usan unas pocas filas del conjunto de prueba solo para *predecir*; no se calcula ninguna métrica.

In [8]:
os.makedirs("../models", exist_ok=True)
joblib.dump(modelo_lineal, MODELO_LINEAL_PATH, compress=3)
joblib.dump(modelo_rf, MODELO_RF_PATH, compress=3)

for nombre, ruta in [("Regresión lineal", MODELO_LINEAL_PATH), ("Random Forest", MODELO_RF_PATH)]:
    print(f"{nombre}: {ruta} ({os.path.getsize(ruta) / 1e6:.1f} MB)")

tam_rf = os.path.getsize(MODELO_RF_PATH) / 1e6
assert tam_rf < 100, "El Random Forest supera el límite de GitHub: reducir n_estimators o subir min_samples_leaf"
if tam_rf > 50:
    print("Aviso: GitHub advierte con archivos de más de 50 MB; considerar reducir el modelo.")

Regresión lineal: ../models/regresion_lineal.joblib (0.0 MB)
Random Forest: ../models/random_forest.joblib (48.2 MB)


In [9]:
muestra = X_test.head(5)

lineal_cargado = joblib.load(MODELO_LINEAL_PATH)
rf_cargado = joblib.load(MODELO_RF_PATH)

# Las predicciones del modelo cargado deben ser idénticas a las del modelo en memoria
assert np.allclose(lineal_cargado.predict(muestra), modelo_lineal.predict(muestra))
assert np.allclose(rf_cargado.predict(muestra), modelo_rf.predict(muestra))
print("Los modelos cargados producen las mismas predicciones que los entrenados.\n")

ejemplo = muestra[["ruta_origen", "empresa", "hora_salida_decimal"]].copy()
ejemplo["pred. regresión lineal (h)"] = np.clip(lineal_cargado.predict(muestra), 0, None).round(2)
ejemplo["pred. random forest (h)"] = rf_cargado.predict(muestra).round(2)
ejemplo

Los modelos cargados producen las mismas predicciones que los entrenados.



,ruta_origen,empresa,hora_salida_decimal,pred. regresión lineal (h),pred. random forest (h)
0,JERICO,TRANSPORTES JERICO,17.000000,3.39,3.42
1,SEGOVIA,TRANSPORTES SEGOVIA,8.250000,5.71,5.79
2,EL CARMEN DE VIBORAL,FLOTA EL CARMEN,19.666667,1.17,1.47
3,CISNEROS,TRANS CISNEROS-ENTRERRIOS,5.500000,1.80,1.75
4,MANIZALES,EMPRESA ARAUCA S.A.,17.500000,5.85,5.24


## 6. Reproducibilidad

- **Semilla fija** (`SEED = 42`) en la separación de datos (notebook 02) y en el Random Forest. La regresión lineal es determinista.
- **Se ejecuta de principio a fin** sin intervención manual. Solo necesita `train.csv.gz` y `test.csv.gz` en `fase-1/data/processed/`, que vienen en el repositorio.
- **Versiones** usadas en esta ejecución (importan al cargar los modelos guardados, que deben abrirse con una versión compatible de scikit-learn):

In [10]:
print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__} | numpy: {np.__version__} | scikit-learn: {sklearn.__version__} | joblib: {joblib.__version__}")

Python: 3.11.9 | pandas: 3.0.6 | numpy: 2.1.3 | scikit-learn: 1.9.1 | joblib: 1.6.0


## Resumen

- **Datos:** `train` y `test` del notebook 02, sin valores faltantes.
- **Modelo 1:** regresión lineal (one-hot de las categóricas y de la hora entera) en un `Pipeline`, guardado en `models/regresion_lineal.joblib`.
- **Modelo 2:** Random Forest (80 árboles, hojas de mínimo 50 viajes, cada árbol con el 50 % de las filas) en un `Pipeline`, guardado en `models/random_forest.joblib`, dentro del límite de tamaño de GitHub.
- **Siguiente paso:** `04-evaluacion.ipynb`, que carga ambos modelos y hace la validación cruzada, la evaluación en el conjunto de prueba, el análisis del error y la comparación (incluido el modelo base).